# Exploratory Data Analysis — Synthetic Example Dataset

> **Synthetic demonstration data.** `synthetic_example_orders.csv` was
> hand-designed purely to demonstrate this project's workflow. It is
> **not real collected data** and must not be presented or analyzed as such.
> See `data/raw/README.md` for its full description.

**Setup** (once, in a fresh environment — see the project README):

```bash
pip install -e ".[notebook]"
```

This notebook then runs **top to bottom** without modification. It:

1. loads the example dataset,
2. shows shape, columns, dtypes and basic statistics,
3. inspects missing values and duplicates,
4. demonstrates validation (default config + stricter example rules),
5. runs the preprocessing workflow,
6. compares the data before and after.

The raw file is only ever read — outputs never overwrite `data/raw/`.

In [1]:
from pathlib import Path

import pandas as pd
import yaml

from src.cleaning.inconsistent_values import normalize_strings
from src.data.load_data import load_csv
from src.data.validate_data import validate_dataframe
from src.pipeline.preprocessing_pipeline import clean, transform

# Locate the project root whether the kernel starts in notebooks/ or the repo root.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config" / "config.yaml").is_file())
config = yaml.safe_load((ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
RAW = ROOT / "data" / "raw" / "synthetic_example_orders.csv"

print("project root :", ROOT)
print("dataset      :", RAW.name, "(synthetic demo data)")
print("imputation   :", config["cleaning"]["missing_values"]["strategy"],
      "| scaling:", config["transformation"]["scaling"]["method"],
      "| encoding:", config["transformation"]["encoding"]["method"])

project root : C:\Users\user\Desktop\reproducible-data-cleaning-and-transformation
dataset      : synthetic_example_orders.csv (synthetic demo data)
imputation   : median | scaling: standard | encoding: onehot


## 1. Load the example dataset

Six columns: free text (`customer_name`), categoricals (`region`, `product`) and numerics (`units`, `unit_price`, `rating`).

In [2]:
df = load_csv(RAW)
df.head(10)

,customer_name,region,product,units,unit_price,rating
0,Alice Smith,North,Widget A,3.0,19.99,4.5
1,alice smith,NORTH,Widget A,3.0,19.99,4.5
2,bob jones,South,Widget A,2.0,19.99,NaN
3,bob jones,South,Widget A,2.0,19.99,NaN
4,carol lee,East,Widget B,4.0,9.99,5.0
5,Carol Lee,east,Gadget C,1.0,149.50,2.5
6,dan brown,West,NaN,2.0,19.99,4.0
7,DAN BROWN,west,Widget A,-1.0,19.99,3.0
8,eve adams,North,Gadget C,3.0,149.50,6.4
9,Eve Adams,north,Widget B,NaN,NaN,3.0


## 2. Shape and columns

In [3]:
print("shape:", df.shape)
print("columns:", list(df.columns))
df.dtypes.to_frame(name="dtype")

shape: (18, 6)
columns: ['customer_name', 'region', 'product', 'units', 'unit_price', 'rating']


,dtype
customer_name,str
region,str
product,str
units,float64
unit_price,float64
rating,float64


## 3. Missing values and duplicates

The file contains missing values in every column except none — nine cells
in total — plus duplicates of two kinds:

* **exact** duplicate rows (byte-identical), and
* rows that only become duplicates **after** string normalization
  (case/whitespace variants such as `' Alice Smith '` vs `'alice smith'`).

That second kind is why the pipeline normalizes *before* dropping
duplicates (`docs/methodology.md`, section 3).

In [4]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "fraction": (df.isna().mean() * 100).round(1).astype(str) + "%",
})
missing = missing[missing["missing"] > 0].sort_values("missing", ascending=False)
display(missing)
print("total missing cells:", int(df.isna().sum().sum()))

,missing,fraction
rating,4,22.2%
customer_name,1,5.6%
region,1,5.6%
product,1,5.6%
units,1,5.6%
unit_price,1,5.6%


total missing cells: 9


In [5]:
print("exact duplicate rows:      ", int(df.duplicated().sum()))
print("duplicates after normalizing:",
      int(normalize_strings(df).duplicated().sum()))

exact duplicate rows:       1
duplicates after normalizing: 2


## 4. Data types and basic statistics

In [6]:
display(df.describe().T)
display(df["region"].value_counts(dropna=False).to_frame("count"))
display(df["product"].value_counts(dropna=False).to_frame("count"))

,count,mean,std,min,25%,50%,75%,max
units,17.0,2.823529,2.377901,-1.00,2.000,2.00,3.00,10.0
unit_price,17.0,47.992353,58.183503,9.99,17.990,19.99,19.99,149.5
rating,14.0,3.992857,1.000357,2.50,3.125,4.00,4.50,6.4


,count
region,
South,4
North,3
East,3
West,2
west,2
NORTH,1
east,1
north,1
NaN,1


,count
product,
Widget A,8
Widget B,5
Gadget C,4
NaN,1


## 5. Validation

**Default config** — `config/config.yaml` ships with all validation rules
empty, so this file passes the gate the runner applies
(`required_columns`, `expected_dtypes`, `ranges` are all optional).

**Stricter example rules** — the dataset deliberately contains two
out-of-range values (`units = -1`, `rating = 6.4`) so validation has
something to catch. These are *planted demonstration violations*, not
typos in the pipeline.

In [7]:
default_problems = validate_dataframe(df, config["validation"])
print("default-config problems:", default_problems or "none - file passes the gate")

default-config problems: none - file passes the gate


In [8]:
strict_rules = {
    "required_columns": list(df.columns),
    "expected_dtypes": {
        "customer_name": "string", "region": "string", "product": "string",
        "units": "numeric", "unit_price": "numeric", "rating": "numeric",
    },
    "ranges": {
        "units": {"min": 1, "max": 50},
        "rating": {"min": 1.0, "max": 5.0},
    },
}
problems = validate_dataframe(df, strict_rules)
print(f"{len(problems)} problem(s) found:")
for p in problems:
    print(" -", p)

2 problem(s) found:
 - Column 'units' fails range check: 1 value(s) below min 1 (e.g. -1.0).
 - Column 'rating' fails range check: 1 value(s) above max 5.0 (e.g. 6.4).


## 6. Preprocessing workflow

Exactly what `scripts/run_pipeline.py` runs per file — but in memory:

`clean()` → drop high-missing columns → normalize strings → impute → deduplicate,
then `transform()` → scale numerics → encode categoricals.

Notes on the actual behavior (see `docs/methodology.md`):

* `mean`/`median` imputation fills **numeric columns only** — text columns
  keep their missing values until encoding (one-hot renders them as an
  all-zero row), which is why *cleaned* below still has missing cells.
* Normalization strips **leading/trailing** whitespace and applies
  `normalize_case: lower` to string values.

In [9]:
cleaned = clean(df.copy(), config)      # raw df is never modified
processed = transform(cleaned, config)

print("raw      :", df.shape)
print("cleaned  :", cleaned.shape, "(2 duplicate rows removed)")
print("processed:", processed.shape)

raw      : (18, 6)
cleaned  : (16, 6) (2 duplicate rows removed)
processed: (16, 16)


## 7. Before and after

In [10]:
def metrics(frame):
    return {
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing cells": int(frame.isna().sum().sum()),
        "duplicate rows": int(frame.duplicated().sum()),
        "numeric cols": len(frame.select_dtypes(include="number").columns),
        "text/object cols": len(frame.select_dtypes(include=["object", "string"]).columns),
    }

comparison = pd.DataFrame({
    "raw": metrics(df),
    "cleaned": metrics(cleaned),
    "processed": metrics(processed),
}).T
display(comparison)

print("missing cells: raw 9 -> cleaned 3 (numeric filled by median; the three")
print("text NaNs remain) -> processed 0 (one-hot encodes them as all-zero rows).")

,rows,columns,missing cells,duplicate rows,numeric cols,text/object cols
raw,18,6,9,1,3,3
cleaned,16,6,3,0,3,3
processed,16,16,0,0,16,0


missing cells: raw 9 -> cleaned 3 (numeric filled by median; the three
text NaNs remain) -> processed 0 (one-hot encodes them as all-zero rows).


In [11]:
# Normalization + dedup in action: the four rows below contain one exact
# duplicate pair and one case/whitespace pair -> only two survive cleaning.
display(df.loc[df.index[:4], ["customer_name", "region", "rating"]])
display(cleaned[["customer_name", "region", "rating"]].head(4))

# Region casing collapses to canonical values:
display(df["region"].value_counts(dropna=False).to_frame("raw counts"))
display(cleaned["region"].value_counts(dropna=False).to_frame("cleaned counts"))

,customer_name,region,rating
0,Alice Smith,North,4.5
1,alice smith,NORTH,4.5
2,bob jones,South,NaN
3,bob jones,South,NaN


,customer_name,region,rating
0,alice smith,north,4.5
2,bob jones,south,4.0
4,carol lee,east,5.0
5,carol lee,east,2.5


,raw counts
region,
South,4
North,3
East,3
West,2
west,2
NORTH,1
east,1
north,1
NaN,1


,cleaned counts
region,
north,4
east,4
west,4
south,3
NaN,1


In [12]:
print("processed columns:", list(processed.columns))
print("missing cells in processed output:", int(processed.isna().sum().sum()))
processed.head(8)

processed columns: ['units', 'unit_price', 'rating', 'customer_name_alice smith', 'customer_name_bob jones', 'customer_name_carol lee', 'customer_name_dan brown', 'customer_name_eve adams', 'customer_name_grace kim', 'region_east', 'region_north', 'region_south', 'region_west', 'product_gadget c', 'product_widget a', 'product_widget b']
missing cells in processed output: 0


,units,unit_price,rating,customer_name_alice smith,customer_name_bob jones,customer_name_carol lee,customer_name_dan brown,customer_name_eve adams,customer_name_grace kim,region_east,region_north,region_south,region_west,product_gadget c,product_widget a,product_widget b
0,0.078865,-0.515335,0.602416,1,0,0,0,0,0,0,1,0,0,0,1,0
2,-0.341750,-0.515335,0.042029,0,1,0,0,0,0,0,0,1,0,0,1,0
4,0.499481,-0.688542,1.162803,0,0,1,0,0,0,1,0,0,0,0,0,1
5,-0.762366,1.727873,-1.639131,0,0,1,0,0,0,1,0,0,0,1,0,0
6,-0.341750,-0.515335,0.042029,0,0,0,1,0,0,0,0,0,1,0,0,0
7,-1.603598,-0.515335,-1.078745,0,0,0,1,0,0,0,0,0,1,0,1,0
8,0.078865,1.727873,2.731886,0,0,0,0,1,0,0,1,0,0,1,0,0
9,-0.341750,-0.515335,-1.078745,0,0,0,0,1,0,0,1,0,0,0,0,1


## Wrap-up

Run the same workflow on disk (raw → `data/interim/` → `data/processed/`):

```bash
python scripts/run_pipeline.py
```

* Remember: the dataset is **synthetic demonstration data**, not real data.
* Documented limitations: `mean`/`median` fill numeric columns only, and
  `read_csv` type inference drops leading zeros from ID-style columns
  (`docs/methodology.md`, sections 1 and 3).